In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pandas_datareader as web
import datetime as dt
import tensorflow as tf
import os
import yfinance as yf
import joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [ ]:
COMPANY = "AAPL"

START_DATE = dt.datetime(2020, 1, 1)
END_DATE = dt.datetime(2023, 8, 1)

DATA_DIR = "data"
DATA_FILE = f"{COMPANY}.csv"

FEATURES = [
    "Open",
    "Close",
    "Low",
    "High",
    "Volume"
]

LOOKBACK = 60
PREDICT_PRICE = 5
TRAIN_RATIO = 0.8

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
csv_path = os.path.join(DATA_DIR, DATA_FILE)

if os.path.exists(csv_path):
    print("Loading local data...")
    data = pd.read_csv(csv_path, index_col=0, parse_dates=True)
else:
    print("Downloading data...")
    data = yf.download(COMPANY, start=START_DATE, end=END_DATE)
    data.to_csv(csv_path)

data = data.dropna()
print(data.shape)
data.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data["Close"])
plt.title(f"{COMPANY} Closing Price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

In [ ]:
data = data[FEATURES]
data.head()

In [ ]:
# feature scaler — scales all 5 input columns together
feature_scaler = MinMaxScaler()
scaled_data = feature_scaler.fit_transform(data)

# target scaler — scales only the Close column (index 1) used for inverse transform
target_scaler = MinMaxScaler()
target_scaler.fit(data[["Close"]])

print("scaled_data shape:", scaled_data.shape)

In [ ]:
X = []
y = []

close_col = FEATURES.index("Close")

for i in range(LOOKBACK, len(scaled_data) - PREDICT_PRICE + 1):
    X.append(scaled_data[i - LOOKBACK : i])
    y.append(scaled_data[i: i+PREDICT_PRICE, close_col ])

X = np.array(X)
y = np.array(y)

print("X:", X.shape)
print("y:", y.shape)

In [ ]:
split_idx = int(len(X) * TRAIN_RATIO)

X_train = X[:split_idx]
X_test = X[split_idx:]

y_train = y[:split_idx]
y_test = y[split_idx:]

print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

In [ ]:
np.savez(
    os.path.join(DATA_DIR, "train_data.npz"),
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

joblib.dump(feature_scaler, os.path.join(DATA_DIR, "feature_scaler.pkl"))
joblib.dump(target_scaler,  os.path.join(DATA_DIR, "target_scaler.pkl"))

print("Saved successfully")

In [ ]:
saved = np.load(os.path.join(DATA_DIR, "train_data.npz"))
print(saved.files)